# LFM Tiling Example

This notebook demonstrates the configuration-driven lunar tiling API. It creates and visualizes two mixed-modality examples on the LTM grid:

1. A product-scoped WAC cube plus the canonical 63-band static cube.
2. A product-scoped NAC cube plus the same static context.

All raster sources use existing, explicitly configured vector indexes and bilinear resampling. Results are returned as `TileCubeRecord` objects, so downstream code uses structured source, zone, zoom, and tile fields instead of parsing output filenames. The legacy notebook remains at `notebooks/toy_model/tiling_example.ipynb` during migration.

## Imports and repository discovery

Run this notebook from the repository's top-level `notebooks/` directory. JupyterHub may expose the clone through `/panfs`; repository discovery normalizes that path to the equivalent `/explore` symlink before importing LFM.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import warnings

from datetime import datetime
from getpass import getuser
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio

warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

In [ ]:
repo_root = Path.cwd().parent
repo_root_str = str(repo_root).replace('/panfs/ccds02/nobackup', '/explore/nobackup')
repo_root = Path(repo_root_str)
NOTEBOOK_DIR = repo_root / "notebooks"

if not (repo_root / "lfm").exists() or not (repo_root / "model").exists():
    raise FileNotFoundError(
        "Cannot find the lfm/ and model/ directories. Run this notebook "
        "from the repository's top-level notebooks/ directory."
    )

# The tiling backend is lfm.model, where lfm is the cloned repository package.
sys.path.insert(0, str(repo_root.parent))

from lfm.model import (
    BandNoDataOverride,
    MINIRF_SOURCE_NODATA,
    MINIRF_SOURCE_NODATA_BANDS,
    STATIC_BAND_NAMES,
    STATIC_OUTPUT_NODATA,
    TileConfig,
    TileSourceConfig,
    create_tiles_for_aoi,
    create_tiles_for_index,
    create_tiles_for_point,
)

print(f"Repository root: {repo_root}")
print("Successfully imported the modern LFM tiling API")

## User configuration

The defaults below use the representative Explore data exercised by the tiling validation suite. Change these paths and selectors for another dataset. Each modality declares its raster directory and existing `.shp` or `.gpkg` index explicitly; tiling never creates or refreshes an index.

The output directory is timestamped so rerunning the notebook does not silently reuse an earlier cube. The AOI intersects two zoom-5 tiles in LTM zone `42N`. NAC coverage is sparse, so the NAC example may have fewer dynamic/static pairs than the WAC example.

In [ ]:
PROJECT_DATA_DIR = Path("/explore/nobackup/projects/lfm")
WAC_DATA_DIR = PROJECT_DATA_DIR / "processed_data/Lunar/LRO_WAC_Pho_Sites"
NAC_DATA_DIR = PROJECT_DATA_DIR / "processed_data/Lunar/LRO_NAC_Pho_Sites"
STATIC_DATA_DIR = PROJECT_DATA_DIR / "staticLinks"

WAC_INDEX = WAC_DATA_DIR / "output_index.shp"
NAC_INDEX = NAC_DATA_DIR / "output_index.shp"
STATIC_INDEX = STATIC_DATA_DIR / "db2.shp"
LOCATION_FIELD = "location"

WAC_PRODUCT_ID = "M1187363083CE"
NAC_PRODUCT_ID = "M1117899885LE"
AOI_BOUNDS = {
    "ul_lat": 1.3,
    "ul_lon": 149.7,
    "lr_lat": 1.1,
    "lr_lon": 149.9,
}
ZOOM_LEVEL = 5
EXPECTED_ZONE = "42N"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
BASE_OUTPUT_DIR = (
    Path("/explore/nobackup/people")
    / getuser()
    / "lfm_notebook_outputs"
    / "tiling"
)
OUTPUT_DIR = BASE_OUTPUT_DIR / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

# Plot WAC VIS band 0, the single NAC image band, and static elevation.
WAC_BAND_NUMBER = 3  # 1-based raster band: first VIS channel after two UV bands
NAC_BAND_NUMBER = 1
STATIC_BAND_TO_PLOT = "lola_kaguya_60mpp_elv"
MAX_PLOT_TILES = 2
RUN_ALTERNATE_QUERIES = False

for label, path in {
    "WAC data directory": WAC_DATA_DIR,
    "NAC data directory": NAC_DATA_DIR,
    "static data directory": STATIC_DATA_DIR,
}.items():
    if not path.is_dir():
        raise FileNotFoundError(f"{label} does not exist: {path}")

for label, path in {
    "WAC index": WAC_INDEX,
    "NAC index": NAC_INDEX,
    "static index": STATIC_INDEX,
}.items():
    if not path.is_file():
        raise FileNotFoundError(f"{label} does not exist: {path}")

print(f"Notebook outputs: {OUTPUT_DIR}")

## Configuration and visualization helpers

`make_static_source` centralizes the canonical 63-band ordering and shared `-32768` output NoData contract. The Mini-RF sentinels remain source-only overrides so they are masked before bilinear interpolation.

The plotting helper follows the datacube inference visualization style: it reads masked raster bands, uses robust display limits, gives every image its own colorbar, and labels columns from structured tile metadata.

In [ ]:
def make_static_source() -> TileSourceConfig:
    return TileSourceConfig(
        name="static",
        data_dir=STATIC_DATA_DIR,
        index_path=STATIC_INDEX,
        location_field=LOCATION_FIELD,
        selection_mode="all_intersecting",
        band_names=STATIC_BAND_NAMES,
        resampling="bilinear",
        output_nodata=STATIC_OUTPUT_NODATA,
        band_nodata_overrides=tuple(
            BandNoDataOverride(
                band_name=name,
                source_value=MINIRF_SOURCE_NODATA,
            )
            for name in MINIRF_SOURCE_NODATA_BANDS
        ),
    )


def tile_key(record) -> tuple[str, int, int, int]:
    return record.zone, record.zoom_level, record.tile_x, record.tile_y


def pair_dynamic_and_static(records, dynamic_source: str):
    records_by_tile = {}
    for record in records:
        records_by_tile.setdefault(tile_key(record), {})[record.source_name] = record
    return [
        (sources[dynamic_source], sources["static"])
        for _, sources in sorted(records_by_tile.items())
        if dynamic_source in sources and "static" in sources
    ]


def print_record_summary(records) -> None:
    for record in records:
        nodata_counts = []
        with rasterio.open(record.path) as source:
            for band_number in range(1, source.count + 1):
                band = source.read(band_number, masked=True)
                nodata_counts.append(int(np.ma.getmaskarray(band).sum()))
        if len(record.band_names) <= 7:
            band_preview = list(record.band_names)
        else:
            band_preview = [
                *record.band_names[:3],
                "...",
                *record.band_names[-3:],
            ]
        print(
            f"{record.source_name:>6} | LTM{record.zone} | z{record.zoom_level} | "
            f"tile=({record.tile_x}, {record.tile_y}) | "
            f"bands={len(record.band_names):>2} | {record.path.name}"
        )
        print(f"         band names: {band_preview}")
        print(
            f"         NoData pixels per band: min={min(nodata_counts):,}, "
            f"max={max(nodata_counts):,}, total={sum(nodata_counts):,}"
        )


def robust_limits(image) -> tuple[float, float]:
    values = np.asarray(image.compressed(), dtype=np.float64)
    values = values[np.isfinite(values)]
    if not values.size:
        return 0.0, 1.0
    lower, upper = np.percentile(values, (2.0, 98.0))
    if np.isclose(lower, upper):
        padding = max(abs(float(lower)) * 0.01, 1.0)
        return float(lower - padding), float(upper + padding)
    return float(lower), float(upper)


def read_record_band(record, *, band_number=None, band_name=None):
    if (band_number is None) == (band_name is None):
        raise ValueError("Provide exactly one of band_number or band_name.")
    if band_name is not None:
        try:
            band_number = record.band_names.index(band_name) + 1
        except ValueError as exc:
            raise KeyError(f"Band {band_name!r} is not present in {record.path}") from exc
    with rasterio.open(record.path) as source:
        image = source.read(band_number, masked=True)
        name = source.tags(band_number).get("Name", f"band_{band_number}")
    return image, name, band_number


def plot_cube_pairs(
    pairs,
    *,
    dynamic_label: str,
    dynamic_band_number: int,
    output_path: Path,
    max_tiles: int = 2,
):
    pairs = list(pairs[:max_tiles])
    if not pairs:
        raise ValueError(f"No {dynamic_label}/static tile pairs are available to plot.")

    figure = plt.figure(
        figsize=(6 * len(pairs), 10),
        constrained_layout=True,
    )
    grid = figure.add_gridspec(
        2,
        2 * len(pairs),
        width_ratios=[value for _ in pairs for value in (1.0, 0.05)],
    )

    for column, (dynamic_record, static_record) in enumerate(pairs):
        dynamic, dynamic_name, dynamic_number = read_record_band(
            dynamic_record,
            band_number=dynamic_band_number,
        )
        static, static_name, static_number = read_record_band(
            static_record,
            band_name=STATIC_BAND_TO_PLOT,
        )
        tile_title = (
            f"LTM{dynamic_record.zone} z{dynamic_record.zoom_level} "
            f"tile ({dynamic_record.tile_x}, {dynamic_record.tile_y})"
        )

        for row, (image, name, number, cmap, label) in enumerate(
            (
                (dynamic, dynamic_name, dynamic_number, "gray", dynamic_label),
                (static, static_name, static_number, "terrain", "STATIC"),
            )
        ):
            axis = figure.add_subplot(grid[row, 2 * column])
            colorbar_axis = figure.add_subplot(grid[row, 2 * column + 1])
            vmin, vmax = robust_limits(image)
            rendered = axis.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
            axis.set_title(f"{tile_title}\n{label} band {number}: {name}")
            axis.set_xlim(-0.5, image.shape[1] - 0.5)
            axis.set_ylim(image.shape[0] - 0.5, -0.5)
            axis.set_aspect("equal", adjustable="box")
            axis.axis("off")
            figure.colorbar(rendered, cax=colorbar_axis)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    figure.suptitle(f"{dynamic_label} + STATIC LTM cubes", y=1.01)
    figure.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved visualization: {output_path}")
    return figure

## Example 1: WAC + STATIC AOI tiling

WAC uses `product_id` selection and preserves its native per-band NoData metadata. Static uses `all_intersecting`, the canonical band order, and standardized output NoData. Source order controls the order of records returned within each tile.

In [ ]:
wac_source = TileSourceConfig(
    name="wac",
    data_dir=WAC_DATA_DIR,
    index_path=WAC_INDEX,
    location_field=LOCATION_FIELD,
    selection_mode="product_id",
    resampling="bilinear",
    preserve_source_nodata=True,
)
wac_static_config = TileConfig(
    output_dir=OUTPUT_DIR / "wac_static",
    zoom_level=ZOOM_LEVEL,
    sources=(wac_source, make_static_source()),
)
wac_static_config

In [ ]:
wac_static_records = create_tiles_for_aoi(
    wac_static_config,
    **AOI_BOUNDS,
    selectors={"wac": WAC_PRODUCT_ID},
)
print_record_summary(wac_static_records)

wac_static_pairs = pair_dynamic_and_static(wac_static_records, "wac")
print(f"WAC/static pairs available for plotting: {len(wac_static_pairs)}")

In [ ]:
wac_figure = plot_cube_pairs(
    wac_static_pairs,
    dynamic_label="WAC",
    dynamic_band_number=WAC_BAND_NUMBER,
    output_path=OUTPUT_DIR / "plots" / "wac_static_cubes.png",
    max_tiles=MAX_PLOT_TILES,
)

## Example 2: NAC + STATIC AOI tiling

The same public API handles NAC without a WAC alias. NAC is declared `required=False` because the selected observation is sparse: an AOI tile without that NAC product is an expected skip, while static context is still produced for every intersecting tile. Plot pairing uses only tiles containing both modalities.

In [ ]:
nac_source = TileSourceConfig(
    name="nac",
    data_dir=NAC_DATA_DIR,
    index_path=NAC_INDEX,
    location_field=LOCATION_FIELD,
    selection_mode="product_id",
    resampling="bilinear",
    preserve_source_nodata=True,
    required=False,
)
nac_static_config = TileConfig(
    output_dir=OUTPUT_DIR / "nac_static",
    zoom_level=ZOOM_LEVEL,
    sources=(nac_source, make_static_source()),
)
nac_static_config

In [ ]:
nac_static_records = create_tiles_for_aoi(
    nac_static_config,
    **AOI_BOUNDS,
    selectors={"nac": NAC_PRODUCT_ID},
)
print_record_summary(nac_static_records)

nac_static_pairs = pair_dynamic_and_static(nac_static_records, "nac")
print(f"NAC/static pairs available for plotting: {len(nac_static_pairs)}")

In [ ]:
nac_figure = plot_cube_pairs(
    nac_static_pairs,
    dynamic_label="NAC",
    dynamic_band_number=NAC_BAND_NUMBER,
    output_path=OUTPUT_DIR / "plots" / "nac_static_cubes.png",
    max_tiles=MAX_PLOT_TILES,
)

## Alternative query entry points

The examples above use a geographic AOI. The same `TileConfig` contract supports a point query and an explicit LTM tile-index query. Set `RUN_ALTERNATE_QUERIES = True` in the user configuration to run the examples below. Separate output directories prevent these calls from overwriting the AOI outputs.

In [ ]:
if RUN_ALTERNATE_QUERIES:
    point_config = TileConfig(
        output_dir=OUTPUT_DIR / "wac_static_point",
        zoom_level=ZOOM_LEVEL,
        sources=wac_static_config.sources,
    )
    point_records = create_tiles_for_point(
        point_config,
        lat=1.2,
        lon=149.8,
        zone=EXPECTED_ZONE,
        selectors={"wac": WAC_PRODUCT_ID},
    )
    print("Point-query records:")
    print_record_summary(point_records)

    index_config = TileConfig(
        output_dir=OUTPUT_DIR / "wac_static_tile_index",
        zoom_level=ZOOM_LEVEL,
        sources=wac_static_config.sources,
    )
    index_records = create_tiles_for_index(
        index_config,
        tile_x=1,
        tile_y=63,
        zone=EXPECTED_ZONE,
        selectors={"wac": WAC_PRODUCT_ID},
    )
    print("Tile-index-query records:")
    print_record_summary(index_records)
else:
    print("Alternate point and tile-index queries are configured but disabled.")

## Outputs

The run directory contains separate WAC/static and NAC/static cube directories plus saved PNG visualizations. Each returned record contains the authoritative modality and tile identity; filenames remain descriptive for human inspection but are not the machine-readable interface.

In [ ]:
print(f"Completed tiling notebook run: {OUTPUT_DIR}")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(OUTPUT_DIR)}")